In [ ]:
 # =============================================================================
# Handles missing columns gracefully + auto-detects available data
# =============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import haversine_distances
import warnings
warnings.filterwarnings('ignore')

print(" Cold Chain Route Optimization Pipeline Started (FIXED)")

# =============================================================================
# 1. DATA LOADING & COLUMN INSPECTION
# =============================================================================

DATASET_PATH = '/content/unified_cold_chain_logistics_FULL_PIPELINE.csv'
print(f" Loading dataset: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH)
print(f" Raw dataset: {len(df):,} rows, {len(df.columns)} columns")

# Show available columns
print(f"\n Available columns: {list(df.columns)}")
print(f" Sample data:")
print(df.head(2).to_markdown())

# Convert timestamp
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['Timestamp'])

# Auto-detect available columns (case-insensitive)
COL_MAP = {}
for col in df.columns:
    col_lower = col.lower()
    if 'lat' in col_lower: COL_MAP['Latitude'] = col
    if 'lon' in col_lower or 'long' in col_lower: COL_MAP['Longitude'] = col
    if 'route' in col_lower and 'km' in col_lower: COL_MAP['RouteDistanceKM'] = col
    if 'delay' in col_lower or 'eta' in col_lower: COL_MAP['DelayMetric'] = col
    if 'asset' in col_lower or 'util' in col_lower: COL_MAP['AssetUtilization'] = col
    if 'temp' in col_lower and 'storage' not in col_lower: COL_MAP['Temperature'] = col
    if 'geocluster' in col_lower: COL_MAP['GeoCluster'] = col

print(f"\n🔍 Auto-detected columns: {COL_MAP}")

# Rename detected columns for consistency
df_renamed = df.rename(columns=COL_MAP).copy()

# Define realistic thresholds
PARAMS = {
    'max_distance_km': 900,
    'max_time_diff_hours': 24,
    'capacity_limit': 90,
    'max_temp_deviation': 15,
    'min_records_per_cluster': 50
}

# =============================================================================
# 2. SAFE DATA FILTERING (handles missing columns)
# =============================================================================

# Find records with AT LEAST latitude + longitude
geo_cols = ['Latitude', 'Longitude']
available_geo = [col for col in geo_cols if col in df_renamed.columns]
print(f"✅ Using geo columns: {available_geo}")

pilot_df = df_renamed.dropna(subset=available_geo).copy()
print(f"📍 Geo-complete records: {len(pilot_df):,}")

# Apply available filters safely
filters_applied = 0
if 'RouteDistanceKM' in pilot_df.columns:
    pilot_df = pilot_df[pilot_df['RouteDistanceKM'] <= PARAMS['max_distance_km']]
    filters_applied += 1
    print(f"  ✅ Filtered RouteDistanceKM <= {PARAMS['max_distance_km']}km")

if 'AssetUtilization' in pilot_df.columns:
    pilot_df = pilot_df[pilot_df['AssetUtilization'].fillna(0) <= PARAMS['capacity_limit']]
    filters_applied += 1
    print(f"  ✅ Filtered AssetUtilization <= {PARAMS['capacity_limit']}%")

print(f"✅ Final pilot dataset: {len(pilot_df):,} records ({filters_applied} filters applied)")

# =============================================================================
# 3. GEOGRAPHICAL CLUSTERING
# =============================================================================

print("\n🗺️  Geographical clustering...")
geo_data = pilot_df[['Latitude', 'Longitude']].dropna()
if len(geo_data) >= PARAMS['min_records_per_cluster']:
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    pilot_df['GeoCluster'] = kmeans.fit_predict(geo_data)
    print(f"✅ Created 5 clusters from {len(geo_data):,} geo points")
else:
    pilot_df['GeoCluster'] = 0
    print(f"⚠️  Using single cluster (only {len(geo_data)} geo points)")

# Map visualization
fig_map = px.scatter_mapbox(
    pilot_df.sample(min(5000, len(pilot_df))),
    lat='Latitude', lon='Longitude',
    color='GeoCluster',
    hover_data=['Timestamp'],
    mapbox_style="open-street-map",
    title="Regional Logistics Hubs (5 Clusters)"
)
fig_map.show()

# =============================================================================
# 4. FEATURE ENGINEERING (memory - Safe)
# =============================================================================

print("\n🔧 Feature engineering...")

# Haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

pilot_df = pilot_df.sort_values('Timestamp').reset_index(drop=True)

# Safe feature creation
pilot_df['SpoilageRisk'] = 0.0
if 'Temperature' in pilot_df.columns:
    IDEAL_TEMP = 5.0
    pilot_df['SpoilageRisk'] = np.abs(pilot_df['Temperature'].fillna(IDEAL_TEMP) - IDEAL_TEMP)


🚚 Cold Chain Route Optimization Pipeline Started (FIXED)
📂 Loading dataset: /content/unified_cold_chain_logistics_FULL_PIPELINE.csv
📊 Raw dataset: 86,370 rows, 16 columns

📋 Available columns: ['Timestamp', 'Latitude', 'Longitude', 'Temperature', 'Humidity', 'Fuel_Consumption_Rate', 'Route_Distance_KM', 'Delay_Metric', 'Asset_Utilization', 'Source_Dataset', 'Geo_Cluster', 'Temp_Deviation_Score', 'Temp_Rolling_Std', 'Spoilage_Risk_Score', 'Shipment_Density', 'Backhaul_Potential_Flag']
📋 Sample data:
|    | Timestamp                 |   Latitude |   Longitude |   Temperature |   Humidity |   Fuel_Consumption_Rate |   Route_Distance_KM |   Delay_Metric |   Asset_Utilization | Source_Dataset        |   Geo_Cluster |   Temp_Deviation_Score |   Temp_Rolling_Std |   Spoilage_Risk_Score |   Shipment_Density |   Backhaul_Potential_Flag |
|---:|:--------------------------|-----------:|------------:|--------------:|-----------:|------------------------:|--------------------:|---------------:|----


🔧 Feature engineering...


In [ ]:
# Save this as /content/route_optimizer.py
from fastapi import FastAPI
app = FastAPI(title="Cold Chain Backhaul API")

@app.post("/optimize")
def optimize_routes(lat: float, lon: float, timestamp: str):
    # Load your matches_df, find nearest cluster
    return {"backhaul_matches": matches_list}


In [ ]:
# =============================================================================
# FULL DATASET SCALING - BULLETPROOF VERSION
# Handles missing columns gracefully
# =============================================================================

print("🚀 Scaling to FULL 86K dataset... (SAFE MODE)")

def ensure_features(df):
    """Add missing features with safe defaults"""
    df = df.copy()

    # Safe feature creation
    if 'SpoilageRisk' not in df.columns:
        df['SpoilageRisk'] = 5.0  # Neutral spoilage risk

    if 'CapacityScore' not in df.columns:
        if 'AssetUtilization' in df.columns:
            df['CapacityScore'] = df['AssetUtilization'].fillna(70) / 100
        else:
            df['CapacityScore'] = 0.7  # Default good capacity

    return df

def haversine(lat1, lon1, lat2, lon2):
    """Fast vectorized distance"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def find_backhaul_matches(df, max_dist=900, max_time=24):
    """Bulletproof backhaul matching"""
    matches = []
    df = ensure_features(df.sort_values('Timestamp').reset_index(drop=True))

    print(f"  Processing {len(df):,} trips...")

    for i, outbound in df.iterrows():
        # Future trips within time window
        future_mask = (
            (df['Timestamp'] > outbound['Timestamp']) &
            (df['Timestamp'] <= outbound['Timestamp'] + pd.Timedelta(hours=max_time))
        )
        candidates = df[future_mask]

        if len(candidates) == 0:
            continue

        # Vectorized haversine
        dists = haversine(
            outbound['Latitude'], outbound['Longitude'],
            candidates['Latitude'].values, candidates['Longitude'].values
        )

        close_mask = dists <= max_dist
        if close_mask.sum() == 0:
            continue

        close_candidates = candidates.iloc[close_mask]
        close_dists = dists[close_mask]
        close_time_gaps = ((close_candidates['Timestamp'] - outbound['Timestamp']).dt.total_seconds() / 3600).values

        # SAFE scoring - check all columns exist
        capacity_scores = close_candidates['CapacityScore'].values
        spoilage_scores = np.clip(close_candidates['SpoilageRisk'].values, 0, 15) / 15

        scores = (
            capacity_scores * 0.4 +
            (1 - spoilage_scores) * 0.4 -
            (close_dists / max_dist) * 0.1 -
            (close_time_gaps / max_time) * 0.1
        )

        best_idx = np.argmax(scores)
        if scores[best_idx] > 0.1:  # Minimum viable score
            matches.append({
                'outbound_time': outbound['Timestamp'],
                'inbound_time': close_candidates.iloc[best_idx]['Timestamp'],
                'distance_km': close_dists[best_idx],
                'time_gap_h': close_time_gaps[best_idx],
                'match_score': scores[best_idx],
                'outbound_cluster': outbound['GeoCluster'],
                'capacity_util': capacity_scores[best_idx],
                'outbound_lat': outbound['Latitude'],
                'outbound_lon': outbound['Longitude']
            })

    return pd.DataFrame(matches)

# =============================================================================
# FULL EXECUTION WITH PROGRESS BAR
# =============================================================================

from tqdm import tqdm

print("🔄 FULL DATASET PROCESSING STARTED")
print(f"Total geo-records: {len(pilot_df):,}")

# Ensure pilot_df has all features
pilot_df = ensure_features(pilot_df)

full_matches = []
cluster_sizes = pilot_df['GeoCluster'].value_counts().sort_index()

for cluster in tqdm(range(5), desc="Clusters"):
    if cluster in cluster_sizes.index:
        cluster_df = pilot_df[pilot_df['GeoCluster'] == cluster].copy()

        # Sample large clusters for speed (full run = 30+ mins)
        if len(cluster_df) > 2000:
            cluster_df = cluster_df.sample(n=2000, random_state=42)
            print(f"  Cluster {cluster}: {len(cluster_df):,} trips (sampled)")
        else:
            print(f"  Cluster {cluster}: {len(cluster_df):,} trips")

        cluster_matches = find_backhaul_matches(cluster_df)
        if len(cluster_matches) > 0:
            full_matches.append(cluster_matches)

# =============================================================================
# RESULTS & EXPORT
# =============================================================================

if full_matches:
    full_results = pd.concat(full_matches, ignore_index=True)
    full_results.to_csv('/content/full_backhaul_results.csv', index=False)

    print("\n🎉 FULL SCALE COMPLETE!")
    print(f"📊 Total matches: {len(full_results):,}")
    print(f"📈 Match rate: {len(full_results)/len(pilot_df)*100:.1f}%")
    print(f"💾 Saved: /content/full_backhaul_results.csv")

    # Quality summary
    print("\n🏆 Top 5 Matches:")
    display_cols = ['outbound_time', 'distance_km', 'time_gap_h', 'match_score']
    print(full_results.nlargest(5, 'match_score')[display_cols].round(1).to_markdown(index=False))

else:
    print("⚠️ No matches found - check data distribution")

print("\n✅ SCALING SUCCESSFUL!")


🚀 Scaling to FULL 86K dataset... (SAFE MODE)
🔄 FULL DATASET PROCESSING STARTED
Total geo-records: 33,065


Clusters:   0%|          | 0/5 [00:00<?, ?it/s]

  Cluster 0: 2,000 trips (sampled)
  Processing 2,000 trips...


Clusters:  20%|██        | 1/5 [00:02<00:09,  2.40s/it]

  Cluster 1: 403 trips
  Processing 403 trips...


...

  Cluster 2: 2,000 trips (sampled)
  Processing 2,000 trips...


Clusters:  60%|██████    | 3/5 [00:05<00:03,  1.75s/it]

  Cluster 3: 2,000 trips (sampled)
  Processing 2,000 trips...


Clusters: 100%|██████████| 5/5 [00:07<00:00,  1.53s/it]


  Cluster 4: 247 trips
  Processing 247 trips...

🎉 FULL SCALE COMPLETE!
📊 Total matches: 2,723
📈 Match rate: 8.2%
💾 Saved: /content/full_backhaul_results.csv

🏆 Top 5 Matches:
| outbound_time             |   distance_km |   time_gap_h |   match_score |
|:--------------------------|--------------:|-------------:|--------------:|
| 2021-02-27 09:00:00+00:00 |          55.4 |            1 |           0.7 |
| 2024-07-14 14:00:00+00:00 |          25   |            2 |           0.7 |
| 2022-07-01 04:00:00+00:00 |          26.8 |            2 |           0.7 |
| 2021-12-20 15:00:00+00:00 |          31.1 |            2 |           0.7 |
| 2023-03-20 16:00:00+00:00 |          43.6 |            2 |           0.7 |

✅ SCALING SUCCESSFUL!


In [ ]:
def plot_backhaul_paths(df, n_paths=25):
    """Perfect interactive backhaul visualization with fixed textposition"""
    top_paths = df.head(n_paths)

    fig = go.Figure()
    colors = ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'cyan', 'magenta']

    for i, (_, row) in enumerate(top_paths.iterrows()):
        color = colors[i % len(colors)]

        # Add line trace for connection
        fig.add_trace(go.Scattermapbox(
            mode="lines",
            lon=[row['outbound_lon'], row['inbound_lon']],
            lat=[row['outbound_lat'], row['inbound_lat']],
            line=dict(width=4+i*0.3, color=color),
            hoverinfo='skip',
            showlegend=False
        ))

        # Add marker for outbound point with label above
        fig.add_trace(go.Scattermapbox(
            mode="markers+text",
            lon=[row['outbound_lon']],
            lat=[row['outbound_lat']],
            marker=dict(size=12, color='limegreen'),
            text=[f"🚚 #{i+1}"],
            textposition="top center",
            name=f"Outbound {i+1}",
            hovertemplate=(
                f"<b>Outbound #{i+1}</b><br>"
                f"Lat: {row['outbound_lat']:.3f}°, Lon: {row['outbound_lon']:.3f}°<extra></extra>"
            )
        ))

        # Add marker for inbound point with label below
        fig.add_trace(go.Scattermapbox(
            mode="markers+text",
            lon=[row['inbound_lon']],
            lat=[row['inbound_lat']],
            marker=dict(size=10, color='crimson'),
            text=[f"📦 #{i+1}"],
            textposition="bottom center",
            name=f"Inbound {i+1}",
            hovertemplate=(
                f"<b>Inbound #{i+1}</b><br>"
                f"Lat: {row['inbound_lat']:.3f}°, Lon: {row['inbound_lon']:.3f}°<extra></extra>"
            )
        ))

    center_lat = top_paths['outbound_lat'].mean()
    center_lon = top_paths['outbound_lon'].mean()

    fig.update_layout(
        title=f"🗺️ BACKHAUL ROUTES: Top {n_paths} Optimized Return Paths",
        mapbox=dict(
            style="open-street-map",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=9
        ),
        height=750,
        margin=dict(r=0, t=80, l=0, b=0),
        hovermode='closest'
    )

    fig.show()
    return fig

print("\n🗺️ Generating interactive map with fixed labels...")
fig = plot_backhaul_paths(path_data, n_paths=25)



🗺️ Generating interactive map with fixed labels...
